# Sector & Industry Relationship Data Preparation

The objective of this notebook is to construct a relational tensor encoding industry-level associations among all qualified market stocks. This tensor serves as an input to the ReRaLSTM model, enabling it to identify stocks that belong to the same industry.

**Output Files**:

* `<MARKET>_industry_relation.npy`: A NumPy array with shape `(num_tickers, num_tickers, industry)`. Each element is a one-hot vector, where a value of `1` at position `[i, j]` denotes that both stock `i` and stock `j` belong to the corresponding industry.


## Imports and paths

In [1]:
import os
import json
import copy
import numpy as np
import pandas as pd
from datetime import datetime

In [2]:
data_path = os.path.join('..', '..', 'data', 'google_finance')
market_name = 'NASDAQ'
relation_path = os.path.join('..', 'data', 'relation', 'relation', 'sector_industry')
industry_ticker_file = os.path.join(relation_path, f'{market_name}_industry_ticker.json')
selected_tickers_file = f'{market_name}_tickers_qualify_dr-0.98_min-5_smooth.csv'

## Load industry mappings
The JSON file containing industry classifications is loaded. This file is structured as a dictionary, where each key represents an industry name (e.g., "Semiconductors") and each value is a list of ticker symbols associated with that industry.

Since relationships are defined between pairs of stocks, only industries with at least two tickers from the selected list are considered valid. These are referred to as "valid" industries. A dictionary, `valid_industry_index`, is then created to map each valid industry name to a unique integer index. This index determines the corresponding channel in the resulting 3D tensor.


In [3]:
selected_tickers = np.genfromtxt(
    os.path.join(data_path, '..', selected_tickers_file),
    dtype=str, delimiter='\t', skip_header=False
)
print('#tickers selected:', len(selected_tickers))

ticker_index = {ticker: idx for idx, ticker in enumerate(selected_tickers)}

#tickers selected: 1026


In [7]:
with open(industry_ticker_file, 'r') as f:
    industry_tickers = json.load(f)

print('#industries:', len(industry_tickers))

#industries: 113


In [ ]:
industry_tickers

In [8]:
valid_industry_index = {}
valid_industry_count = 0
for industry in industry_tickers:
    if len(industry_tickers[industry]) > 1:
        valid_industry_index[industry] = valid_industry_count
        valid_industry_count += 1

print('#valid industries:', len(valid_industry_index))

#valid industries: 96


In [ ]:
valid_industry_index

## Relational tensor construction

The core logic of the notebook involves constructing a 3D NumPy array, `ticker_relation_embedding`, to store multi-hot encoded relationships between ticker pairs.

1. **Initialization:** A zero-initialized tensor of shape (1026, 1026, 97) is created, where the dimensions correspond to (ticker\_i, ticker\_j, industry\_k). The final channel is reserved for a special self-relation flag.

2. **One-Hot Vectors:** An identity matrix serves as a source of one-hot vectors representing individual industries.

3. **Tensor Population:** For each valid industry, all unique pairs of tickers within that industry are identified. For each pair (left, right), the corresponding one-hot vector is inserted at position `tensor[left_idx][right_idx]`, indexed by the industry's position.

4. **Self-Relation:** To support graph-based models, self-relations are explicitly encoded by setting the final element of each diagonal entry `tensor[i][i][-1]` to 1.


In [15]:
# Initialize Embedding Structures
one_hot_industry_embedding = np.identity(valid_industry_count + 1, dtype=int)
ticker_relation_embedding = np.zeros(
    (len(selected_tickers), len(selected_tickers), valid_industry_count + 1), dtype=int
)
ticker_relation_embedding.shape

(1026, 1026, 97)

In [16]:
print('Single relation vector shape:', ticker_relation_embedding[0][0].shape)

Single relation vector shape: (97,)


In [20]:
# Assign Industry Embeddings to Pairs
for industry, ind_index in valid_industry_index.items():
    cur_ind_tickers = industry_tickers[industry]
    if len(cur_ind_tickers) <= 1:
        continue
    for i in range(len(cur_ind_tickers)):
        left = cur_ind_tickers[i]
        if left not in ticker_index:
            continue
        left_idx = ticker_index[left]
        ticker_relation_embedding[left_idx][left_idx] = copy.copy(one_hot_industry_embedding[ind_index])
        ticker_relation_embedding[left_idx][left_idx][-1] = 1
        for j in range(i + 1, len(cur_ind_tickers)):
            right = cur_ind_tickers[j]
            if right not in ticker_index:
                continue
            right_idx = ticker_index[right]
            ticker_relation_embedding[left_idx][right_idx] = copy.copy(one_hot_industry_embedding[ind_index])
            ticker_relation_embedding[right_idx][left_idx] = copy.copy(one_hot_industry_embedding[ind_index])


In [21]:
# Ensure Self-Relations Exist
for i in range(len(selected_tickers)):
    ticker_relation_embedding[i][i][-1] = 1

print('Final relation tensor shape:', ticker_relation_embedding.shape)

Final relation tensor shape: (1026, 1026, 97)


In [ ]:
ticker_relation_embedding

In [ ]:
# Save Embedding Matrix
#output_file = f'{market_name}_industry_relation.npy'
#np.save(output_file, ticker_relation_embedding)
#print(f'Saved relation embedding to: {output_file}')
